# Graph Comparison and Canonical Hashing — User Guide

Two graphs can represent the exact same information while taking different forms, primarily by using different blank node labels. Blank nodes are arbitrary identifiers and not part of the data of the graph.

`.isomorphic()` checks for graph equivalence correctly under RDF-1.2.

`to_canonical_nquads()` and `rdfc10_hash()` implement the [RDFC-1.0] canonicalization algorithm: a deterministic, blank-node-label-independent hash/serialization of a graph, so two isomorphic graphs always hash identically regardless of how their blank nodes happen to be labeled. `to_canonical_nquads()` returns a string which represents the graph, and `rdfc10_hash()` returns a hex-digest string (64 characters for the default `sha256`; 96 for `sha384`). Note that RDFC-1.0 has not been updated for RDF 1.2, so different graph processors may compute different values than StarLayerGraph does.

`to_canonical_nquads()` and `rdfc10_hash()` can be used to determine whether a graph has changed.

## How to run this notebook

See [Getting Started](01-getting-started.ipynb) if you haven't installed StarLayer yet. Run cells from top to bottom — later cells reuse variables from earlier ones.

In [1]:
from starlayer import StarLayerGraph, Namespace, Literal, BNode
from starlayergraph.compare import isomorphic
from starlayergraph.rdfc import rdfc10_hash, to_canonical_nquads

EX = Namespace("http://example.org/")

## `isomorphic()` and `rdfc10_hash()`

In [2]:
g1 = StarLayerGraph()
g1.bind("ex", EX)
b1 = BNode()
g1.add((EX.alice, EX.knows, b1))
g1.add((b1, EX.name, Literal("someone")))

# same shape, deliberately different (fresh, unrelated) blank node label
g2 = StarLayerGraph()
g2.bind("ex", EX)
b2 = BNode()
g2.add((EX.alice, EX.knows, b2))
g2.add((b2, EX.name, Literal("someone")))

print("isomorphic despite different blank node labels:", isomorphic(g1, g2))
print("hash g1:", rdfc10_hash(g1))
print("hash g2:", rdfc10_hash(g2))
print("hashes equal:", rdfc10_hash(g1) == rdfc10_hash(g2))

g3 = StarLayerGraph()
g3.bind("ex", EX)
g3.add((EX.alice, EX.knows, EX.bob))

print()
print("isomorphic to a graph with genuinely different data:", isomorphic(g1, g3))

# modified g1: same triples as g1, plus one additional triple involving a
# new blank node - a genuine content difference, not just a relabeling
g1_modified = StarLayerGraph()
g1_modified.bind("ex", EX)
g1_modified.add((EX.alice, EX.knows, b1))
g1_modified.add((b1, EX.name, Literal("someone")))
g1_modified.add((EX.alice, EX.worksAt, BNode()))

print()
print("isomorphic to g1 plus one extra blank-node triple:", isomorphic(g1, g1_modified))
print("hashes equal:", rdfc10_hash(g1) == rdfc10_hash(g1_modified))

isomorphic despite different blank node labels: True
hash g1: 236939de1885b1e84eef000f98a0803aeacd2611a57fe3cdac66ae4b9cc3cc1c
hash g2: 236939de1885b1e84eef000f98a0803aeacd2611a57fe3cdac66ae4b9cc3cc1c
hashes equal: True

isomorphic to a graph with genuinely different data: False

isomorphic to g1 plus one extra blank-node triple: False
hashes equal: False


In [3]:
# to_canonical_nquads() is the canonical serialization the hash is derived from -
# blank nodes get deterministic c14n labels instead of their original arbitrary ones.
print(to_canonical_nquads(g1))

print()
print("g1_modified:")
print(to_canonical_nquads(g1_modified))




<http://example.org/alice> <http://example.org/knows> _:c14n0 .
_:c14n0 <http://example.org/name> "someone" .


g1_modified:
<http://example.org/alice> <http://example.org/knows> _:c14n1 .
<http://example.org/alice> <http://example.org/worksAt> _:c14n0 .
_:c14n1 <http://example.org/name> "someone" .



## Checking a graph against a known hash

A hash computed once and stored separately becomes a record of the state of the graph. Later, recomputing the hash from the current graph and comparing it against the stored hash value answers the question "is this still the same data?" without needing to keep a full copy of the original around.

However, if the graph has changed, the hash cannot tell you what changed. Only that something changed.

In [4]:
b1_current = BNode()
g_current = StarLayerGraph()
g_current.bind("ex", EX)
g_current.add((EX.alice, EX.knows, b1_current))
g_current.add((b1_current, EX.name, Literal("someone")))

# a hash committed elsewhere (computed once, stored/shared separately) -
# hard-coded here to stand in for that; it's the same value printed as
# "hash g1" above, since g_current starts out with the same shape as g1
committed_hash = "236939de1885b1e84eef000f98a0803aeacd2611a57fe3cdac66ae4b9cc3cc1c"

print("matches committed hash:", rdfc10_hash(g_current) == committed_hash)

# the graph changes...
g_current.add((EX.alice, EX.worksAt, BNode()))

print("matches committed hash after a change:", rdfc10_hash(g_current) == committed_hash)

matches committed hash: True
matches committed hash after a change: False


## Further Reading

1. **[Getting Started](01-getting-started.ipynb)** — install, first parse, first query, first validate.
2. **[Graphs](02-graphs.ipynb)** — `TripleTerm`/`DirLangString` semantics, Turtle 1.2 reification syntax.
4. **[SHACL shapes](04-shacl-shapes.ipynb)**
   - 4.f **[SHACL subgraph extraction](04f-shacl-subgraph-extraction.ipynb)** — builds a commit/verify workflow directly on `rdfc10_hash()`/`isomorphic()`.
5. **Other**
   - 5.d **Canonical hashing and graph comparison** — this guide.